# Study 2: reward comparison

This notebook compares local, global, and mixed rewards under the current multi-pollutant environment. The Q-learning update is in `train.py`; evaluation, settings, and result inspection stay in the notebook.

The model values are synthetic and exploratory. This notebook compares reward modes under the current environment; earlier notebooks also document previous design stages.

## Settings

In [1]:
import json
import pandas as pd

from study_2_economic_sustainability.config import EnvironmentConfig, TrainingConfig
from study_2_economic_sustainability.train import (train_agents, save_q_tables, load_q_tables)
from study_2_economic_sustainability.environment import EconomicSustainabilityEnvironment
from study_2_economic_sustainability.evaluation import run_episode

In [2]:
REWARD_MODES = ("local", "global", "mixed")
RUN_TRAINING = True
environment_config = EnvironmentConfig()
training_config = TrainingConfig()
TRAINING_SEEDS = training_config.training_seeds
EVALUATION_SEEDS = training_config.evaluation_seeds

settings = {
    "environment": environment_config.__dict__,
    "training": training_config.__dict__,
    "reward_modes": REWARD_MODES,
}

with open(f"../results/settings.json", "w", encoding="utf-8") as file:
    json.dump(settings, file, indent=2)
    file.write("\n")

## Train the reward conditions

- Each training seed creates four independent Q-tables.
- All reward modes use the same training seeds and settings.
- The saved Q-tables are kept together in one JSON file.

In [3]:
if RUN_TRAINING:
    trained_policies = {}

    for reward_mode in REWARD_MODES:
        for training_seed in TRAINING_SEEDS:
            agents = train_agents(reward_mode, training_seed, environment_config, training_config)
            trained_policies[(reward_mode, training_seed)] = agents
            print(f"Trained {reward_mode} reward, seed {training_seed}")

    save_q_tables(trained_policies, "../results", notebook_number=3)
else:
    trained_policies = load_q_tables(REWARD_MODES, TRAINING_SEEDS, training_config, "../results", notebook_number=3)

Trained local reward, seed 11
Trained local reward, seed 23
Trained local reward, seed 37
Trained local reward, seed 41
Trained local reward, seed 53
Trained global reward, seed 11
Trained global reward, seed 23
Trained global reward, seed 37
Trained global reward, seed 41
Trained global reward, seed 53
Trained mixed reward, seed 11
Trained mixed reward, seed 23
Trained mixed reward, seed 37
Trained mixed reward, seed 41
Trained mixed reward, seed 53


## Evaluate the trained agents

In [4]:
def evaluate_episode(environment, agents, seed):
    pollutant_totals = {}
    pollutant_maxima = {}
    gpi_total = 0.0
    prevention_spending = 0.0
    cleanup_spending = 0.0

    for pollutant in environment.config.pollutants:
        pollutant_totals[pollutant] = 0.0
        pollutant_maxima[pollutant] = 0.0

    transitions, terminated, truncated = run_episode(environment, agents, seed)

    for transition in transitions:
        info = transition["info"]
        gpi_total += info["global_pollution_index"]
        prevention_spending += sum(info["prevention_costs_paid"])
        cleanup_spending += sum(info["cleanup_costs_paid"])

        for pollutant in environment.config.pollutants:
            for region in environment.config.regions:
                load = info["pollutant_loads"][pollutant][region]
                pollutant_totals[pollutant] += load
                pollutant_maxima[pollutant] = max(pollutant_maxima[pollutant], load)

    episode_rows = {
        "seed": seed,
        "steps": environment.step_count,
        "failure": terminated,
        "mean_gpi": gpi_total / environment.step_count,
        "final_mean_edi": sum(environment.development_indices) / 4,
        "final_currency": sum(environment.currency_balances) / 4,
        "prevention_spending": prevention_spending,
        "cleanup_spending": cleanup_spending,
    }

    for pollutant in environment.config.pollutants:
        episode_rows[f"mean_{pollutant}"] = (pollutant_totals[pollutant] / (environment.step_count * 4))
        episode_rows[f"maximum_{pollutant}"] = pollutant_maxima[pollutant]

    return episode_rows

In [5]:
evaluation_rows = []

for (reward_mode, training_seed), agents in trained_policies.items():
    environment = EconomicSustainabilityEnvironment(reward_mode, environment_config)

    for evaluation_seed in EVALUATION_SEEDS:
        row = evaluate_episode(environment, agents, evaluation_seed)
        row["reward_mode"] = reward_mode
        row["training_seed"] = training_seed
        evaluation_rows.append(row)

evaluations = pd.DataFrame(evaluation_rows)
evaluations.to_csv(f"../results/evaluation_episodes.csv", index=False)
evaluations.head()

,seed,steps,failure,mean_gpi,final_mean_edi,final_currency,prevention_spending,cleanup_spending,mean_air,maximum_air,mean_water,maximum_water,mean_soil,maximum_soil,mean_noise,maximum_noise,reward_mode,training_seed
0,200000,200,False,7.838992,77.5,2208.375,11510.0,19800.0,6.438070,14.306458,7.973127,27.205133,11.957166,34.327585,2.044380,4.961862,local,11
1,200001,200,False,7.240554,77.5,2435.375,12140.0,21315.0,6.316295,14.306711,7.311853,26.398968,10.552988,26.943587,1.980481,4.961820,local,11
2,200002,200,False,7.500202,77.5,2521.125,12460.0,19645.0,6.403062,14.327334,7.330140,20.516165,11.429097,25.529779,2.028136,4.961882,local,11
3,200003,200,False,7.467814,77.5,2429.375,11960.0,21050.0,6.401552,14.327401,7.583293,28.199401,11.005868,29.551997,2.008158,4.961882,local,11
4,200004,200,False,7.554858,77.5,2390.750,11150.0,20560.0,6.498530,14.030774,7.514438,22.011038,11.281705,31.571326,2.056152,4.947723,local,11


## Compare outcomes

In [6]:
per_training_seed = evaluations.groupby(
    ["reward_mode", "training_seed"], sort=False).agg(
    failure_fraction=("failure", "mean"),
    mean_steps=("steps", "mean"),
    mean_gpi=("mean_gpi", "mean"),
    final_edi=("final_mean_edi", "mean"),
    prevention_spending=("prevention_spending", "mean"),
    cleanup_spending=("cleanup_spending", "mean"),
    )

display(per_training_seed.round(2))

failure_fraction  mean_steps  mean_gpi  final_edi  \
reward_mode training_seed                                                      
local       11                          0.0       200.0      8.43      77.96   
            23                          0.0       200.0     17.04      85.82   
            37                          0.0       200.0      7.23      62.56   
            41                          0.0       200.0      7.75      70.00   
            53                          0.0       200.0     20.03      92.61   
global      11                          0.0       200.0     16.00      70.00   
            23                          0.0       200.0      8.89      77.50   
            37                          0.0       200.0     17.89      72.53   
            41                          0.0       200.0     20.92      75.70   
            53                          0.0       200.0     10.32      70.00   
mixed       11                          0.0       200.0     16.65      70.00   
            23                          0.0       200.0     16.10      88.85   
            37                          0.0       200.0     13.37      77.80   
            41                          0.0       200.0     20.38      87.85   
            53                          0.0       200.0     15.54      84.94   

                           prevention_spending  cleanup_spending  
reward_mode training_seed                                         
local       11                        11775.25          19696.75  
            23                         4608.75          10328.25  
            37                        12239.00          10411.25  
            41                         9281.25          12702.25  
            53                        13140.00           6900.50  
global      11                         7150.75           7321.75  
            23                         5743.75          15106.00  
            37                         9293.25           7152.00  
            41                        16999.75           3821.75  
            53                         6454.00          12743.00  
mixed       11                        11146.25           6092.75  
            23                         7051.00           9610.75  
            37                         6531.75          12995.00  
            41                         6644.00           9242.25  
            53                        20657.50           5553.50

- Each row summarizes the held-out episodes for one training seed. Compare reward modes across these seed-level results.
- GPI summarizes the normalized pollution types. The table above also reports the four types separately.
- These results describe behavior under the current synthetic assumptions; they do not establish that one reward is best in a real system.

In [7]:
reward_summary = per_training_seed.groupby("reward_mode", sort=False).agg(
    failure_mean=("failure_fraction", "mean"),
    failure_sd=("failure_fraction", "std"),
    gpi_mean=("mean_gpi", "mean"),
    gpi_sd=("mean_gpi", "std"),
    final_edi_mean=("final_edi", "mean"),
    final_edi_sd=("final_edi", "std"),
    prevention_spending=("prevention_spending", "mean"),
    cleanup_spending=("cleanup_spending", "mean"),
    )

display(reward_summary.round(2))

,failure_mean,failure_sd,gpi_mean,gpi_sd,final_edi_mean,final_edi_sd,prevention_spending,cleanup_spending
reward_mode,,,,,,,,
local,0.0,0.0,12.09,5.99,77.79,12.01,10208.85,12007.80
global,0.0,0.0,14.80,5.08,73.15,3.38,9128.30,9228.90
mixed,0.0,0.0,16.41,2.55,81.89,7.93,10406.10,8698.85


### What the comparison shows

- None of the reward modes failed in the 100 held-out episodes. Mixed reward reached the highest mean final EDI (81.9), while local reward had the lowest mean GPI (12.1).
- Global reward had the lowest combined prevention and cleanup spending (about 18,357), but also the lowest mean final EDI (73.2). Mixed reward spent about 19,105 and local about 22,217.